# UAH Full Dataset Pipeline

## Objective

This notebook automatically processes every driving session in the UAH-DriveSet dataset.

For every trip it:

- Loads raw sensor data
- Synchronizes sensors
- Generates engineered features
- Applies sliding window feature extraction
- Extracts labels
- Combines everything into a single machine learning dataset

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
dataset_path = "/content/drive/MyDrive/TOGG_Driver_Risk_Project/data/raw/UAH-DRIVESET-v1"

print(dataset_path)

/content/drive/MyDrive/TOGG_Driver_Risk_Project/data/raw/UAH-DRIVESET-v1


In [4]:
drivers = sorted([
    d
    for d in os.listdir(dataset_path)
    if os.path.isdir(os.path.join(dataset_path, d))
    and d.startswith("D")
])

drivers

['D1', 'D2', 'D3', 'D4', 'D5', 'D6']

In [6]:
trip_summary = {}

for driver in drivers:

    driver_path = os.path.join(dataset_path, driver)

    trips = sorted([
        trip
        for trip in os.listdir(driver_path)
        if os.path.isdir(os.path.join(driver_path, trip))
    ])

    trip_summary[driver] = trips

    print(f"\n{driver}")

    for trip in trips:
        print("   ", trip)


D1
    20151110175712-16km-D1-NORMAL1-SECONDARY
    20151110180824-16km-D1-NORMAL2-SECONDARY
    20151111123124-25km-D1-NORMAL-MOTORWAY
    20151111125233-24km-D1-AGGRESSIVE-MOTORWAY
    20151111132348-25km-D1-DROWSY-MOTORWAY
    20151111134545-16km-D1-AGGRESSIVE-SECONDARY
    20151111135612-13km-D1-DROWSY-SECONDARY

D2
    20151120131714-26km-D2-NORMAL-MOTORWAY
    20151120133502-26km-D2-AGGRESSIVE-MOTORWAY
    20151120135152-25km-D2-DROWSY-MOTORWAY
    20151120160904-16km-D2-NORMAL1-SECONDARY
    20151120162105-17km-D2-NORMAL2-SECONDARY
    20151120163350-16km-D2-AGGRESSIVE-SECONDARY
    20151120164606-16km-D2-DROWSY-SECONDARY

D3
    20151126110502-26km-D3-NORMAL-MOTORWAY
    20151126113754-26km-D3-DROWSY-MOTORWAY
    20151126124208-16km-D3-NORMAL1-SECONDARY
    20151126125458-16km-D3-NORMAL2-SECONDARY
    20151126130707-16km-D3-AGGRESSIVE-SECONDARY
    20151126132013-17km-D3-DROWSY-SECONDARY
    20151126134736-26km-D3-AGGRESSIVE-MOTORWAY

D4
    20151203171800-16km-D4-NORMAL1-SECO

In [7]:
def parse_trip_info(trip_name):

    parts = trip_name.split("-")

    return {
        "date": parts[0],
        "distance": parts[1],
        "driver": parts[2],
        "behavior": parts[3],
        "road_type": parts[4]
    }

In [8]:
example_trip = trip_summary["D1"][0]

info = parse_trip_info(example_trip)

info

{'date': '20151110175712',
 'distance': '16km',
 'driver': 'D1',
 'behavior': 'NORMAL1',
 'road_type': 'SECONDARY'}

In [9]:
def simplify_behavior(label):

    if "NORMAL" in label:
        return "NORMAL"

    if "AGGRESSIVE" in label:
        return "AGGRESSIVE"

    if "DROWSY" in label:
        return "DROWSY"

    return label

In [10]:
print(simplify_behavior("NORMAL1"))
print(simplify_behavior("NORMAL2"))
print(simplify_behavior("AGGRESSIVE"))
print(simplify_behavior("DROWSY"))

NORMAL
NORMAL
AGGRESSIVE
DROWSY
